In [1]:
# Get the sqlite database path
#| echo: false

from pathlib import Path

import pandas as pd
from great_tables import GT
from great_tables import html
from sqlalchemy import create_engine
from sqlalchemy import text, bindparam

from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.constants import LARGE_ACUTE_PROVIDER_CODES, TOTAL_ONLY_TREATMENT_CODES
from nhs_waiting_lists.constants import proj_db_path, DB_FILE
from nhs_waiting_lists.utils.xdg import XDGBasedir
from nhs_waiting_lists.constants import ALL_PROVIDER_CODES

import nhs_waiting_lists as nhs
project_root = Path(XDGBasedir.get_data_dir(__app_name__))

DB_PATH = project_root / proj_db_path / DB_FILE
FILES_DIR = project_root / "files"

engine = create_engine(f"sqlite:///{DB_PATH}")

# Get the 23 large acute trust provider codes, identified by the ranking table csv
PROVIDER_CODES = LARGE_ACUTE_PROVIDER_CODES

# Get the C_999 meta-treatment code which aggregates all other treatment codes
TREATMENT_CODES = TOTAL_ONLY_TREATMENT_CODES
# TREATMENT_CODES = ALL_TREATMENT_CODES

#
# provider_code = "RAJ"
# treatment_code = "C_110"

In [2]:
#| echo: false
#| output: false

## Retrieve all the provider rtt summary data

start_period = "2024-10"
end_period = "2025-09"

consolidated_df = nhs.get_consolidated_df(
    start_period,
    end_period,
    ALL_PROVIDER_CODES,
    TREATMENT_CODES,
)

consolidated_df.head()


db_path: /home/tomhodder/.local/state/nhs_waiting_lists/db/nhs_waiting_lists.db


,period,provider,treatment,untreated,new_periods,incomplete,incomplete_prev,incomplete_diff,completed,treated,provider_name,provider_type,provider_subtype,total_treatable,incomplete_expected
0,2024-10-01,R0A,C_999,4123,34861,197984,195516,2468,36516,36516,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,230377,193861
1,2024-11-01,R0A,C_999,3283,32231,200564,197984,2580,32934,32934,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,230215,197281
2,2024-12-01,R0A,C_999,-662,30177,200441,200564,-123,29638,29638,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,230741,201103
3,2025-01-01,R0A,C_999,-2508,36145,199390,200441,-1051,34688,34688,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,236586,201898
4,2025-02-01,R0A,C_999,-4204,34228,197960,199390,-1430,31454,31454,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,233618,202164


In [3]:
#| echo: false
#| output: false

other_trusts = consolidated_df.query("provider != 'RM1'")

other_trusts

,period,provider,treatment,untreated,new_periods,incomplete,incomplete_prev,incomplete_diff,completed,treated,provider_name,provider_type,provider_subtype,total_treatable,incomplete_expected
0,2024-10-01,R0A,C_999,4123,34861,197984,195516,2468,36516,36516,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,230377,193861
1,2024-11-01,R0A,C_999,3283,32231,200564,197984,2580,32934,32934,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,230215,197281
2,2024-12-01,R0A,C_999,-662,30177,200441,200564,-123,29638,29638,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,230741,201103
3,2025-01-01,R0A,C_999,-2508,36145,199390,200441,-1051,34688,34688,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,236586,201898
4,2025-02-01,R0A,C_999,-4204,34228,197960,199390,-1430,31454,31454,Manchester University NHS Foundation Trust,Acute trust,Acute - Teaching,233618,202164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,2025-05-01,RYR,C_999,-1356,20604,112989,115225,-2236,21484,21484,University Hospitals Sussex NHS Foundation Trust,Acute trust,Acute - Teaching,135829,114345
1601,2025-06-01,RYR,C_999,838,22184,113130,112989,141,22881,22881,University Hospitals Sussex NHS Foundation Trust,Acute trust,Acute - Teaching,135173,112292
1602,2025-07-01,RYR,C_999,184,23462,114891,113130,1761,21885,21885,University Hospitals Sussex NHS Foundation Trust,Acute trust,Acute - Teaching,136592,114707
1603,2025-08-01,RYR,C_999,-253,19386,114712,114891,-179,19312,19312,University Hospitals Sussex NHS Foundation Trust,Acute trust,Acute - Teaching,134277,114965


In [4]:
#| echo: false
#| output: false

raj = consolidated_df.query("provider == 'RM1'")

raj_result: pd.DataFrame = raj.groupby(['period'])[
    [
        'incomplete_prev',
        'new_periods',
        'completed',
        "total_treatable",
        'untreated',
        "incomplete_expected",
        "incomplete",
    ]].sum().reset_index()

raj_result["unexplained"] = raj_result["untreated"]
raj_result["unexplained_pct"] = raj_result["untreated"] / raj_result["total_treatable"]

raj_result[["period", "unexplained_pct", "incomplete", "unexplained", "total_treatable", "unexplained_pct"]]


,period,unexplained_pct,incomplete,unexplained,total_treatable,unexplained_pct
0,2024-10-01,-0.029542,84074,-3028,102497,-0.029542
1,2024-11-01,-0.025314,83094,-2543,100458,-0.025314
2,2024-12-01,-0.018346,83061,-1793,97734,-0.018346
3,2025-01-01,-0.023156,82750,-2322,100276,-0.023156
4,2025-02-01,-0.020641,82337,-2032,98444,-0.020641
5,2025-03-01,-0.029583,81820,-2934,99178,-0.029583
6,2025-04-01,-0.032187,80247,-3139,97523,-0.032187
7,2025-05-01,-0.019311,80433,-1852,95906,-0.019311
8,2025-06-01,-0.018866,80831,-1828,96893,-0.018866
9,2025-07-01,-0.013508,81837,-1323,97940,-0.013508


$residual = (NewPeriodsObserved - NewPeriodsExpected) / NewPeriodsObserved$

## Mid and South Essex NHS Foundation Trust

Applying the same methodology to Mid and South Essex NHS Foundation Trust over the same reporting period produces materially different results, as shown in Table 3.


In [5]:
#| echo: false
#| output: true

(raj_result[
    [
        "period",
        "incomplete",
        "incomplete_expected",
        "unexplained",
        "total_treatable",
        "unexplained_pct",
    ]].style.relabel_index(
    [
        "",
        'Observed<br>Incomplete<br>Pathways',
        'Expected<br>Incomplete<br>Pathways',
        'Unreported<br>Change',
        'Treatable<br>Pathways',
        'Unreported<br>%'
    ], axis=1).hide(axis='index')
.format(precision=3, thousands=",", decimal=".")
.format('{:.2%}', subset=["unexplained_pct"])
.format(lambda v: v.strftime("%Y-%m"), subset=["period"])
.set_table_styles([
    {'selector': 'th.col_heading', 'props': 'text-align: center;'},
    # {'selector': 'th.col_heading.level0', 'props': 'font-size: 1.5em;'},
    # {'selector': 'td', 'props': 'text-align: center; font-weight: bold;'},
], overwrite=False)
)

,ObservedIncompletePathways,ExpectedIncompletePathways,UnreportedChange,TreatablePathways,Unreported%
2024-10,"84,074","87,102","-3,028","102,497",-2.95%
2024-11,"83,094","85,637","-2,543","100,458",-2.53%
2024-12,"83,061","84,854","-1,793","97,734",-1.83%
2025-01,"82,750","85,072","-2,322","100,276",-2.32%
2025-02,"82,337","84,369","-2,032","98,444",-2.06%
2025-03,"81,820","84,754","-2,934","99,178",-2.96%
2025-04,"80,247","83,386","-3,139","97,523",-3.22%
2025-05,"80,433","82,285","-1,852","95,906",-1.93%
2025-06,"80,831","82,659","-1,828","96,893",-1.89%
2025-07,"81,837","83,160","-1,323","97,940",-1.35%


## References